<a href="https://colab.research.google.com/github/ml-da/ab_tests/blob/main/A_B_tests.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest
from itertools import chain, combinations
import warnings
from google.colab import auth
from google.cloud import bigquery

auth.authenticate_user()
project_id = 'data-analytics-mate'
client = bigquery.Client(project=project_id)

#QUERY

In [ ]:
query = """
WITH
  session_info AS (
    SELECT
      s.date,
      s.ga_session_id,
      sp.country,
      sp.device,
      sp.continent,
      sp.channel,
      ab.test,
      ab.test_group
    FROM `data-analytics-mate.DA.ab_test` ab
    JOIN data-analytics-mate.DA.session s
      ON ab.ga_session_id = s.ga_session_id
    JOIN data-analytics-mate.DA.session_params sp
      ON s.ga_session_id = sp.ga_session_id
  ),
  session_with_orders AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      COUNT(DISTINCT o.ga_session_id) AS session_with_orders
    FROM data-analytics-mate.DA.order o
    JOIN session_info si
      ON o.ga_session_id = si.ga_session_id
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group
  ),
  events AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      ep.event_name,
      COUNT(ep.ga_session_id) AS event_cnt
    FROM data-analytics-mate.DA.event_params ep
    JOIN session_info si
      ON ep.ga_session_id = si.ga_session_id
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      ep.event_name
  ),
  session AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      COUNT(DISTINCT si.ga_session_id) AS session_cnt
    FROM session_info si
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group
  ),
  account AS (
    SELECT
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group,
      COUNT(DISTINCT acs.ga_session_id) AS new_account_cnt
    FROM data-analytics-mate.DA.account_session acs
    JOIN session_info si
      ON acs.ga_session_id = si.ga_session_id
    GROUP BY
      si.date,
      si.country,
      si.device,
      si.continent,
      si.channel,
      si.test,
      si.test_group
  )
SELECT
  so.date,
  so.country,
  so.device,
  so.continent,
  so.channel,
  so.test,
  so.test_group,
  'session_with_orders' AS event_name,
  so.session_with_orders AS value
FROM session_with_orders so
UNION ALL
SELECT
  e.date,
  e.country,
  e.device,
  e.continent,
  e.channel,
  e.test,
  e.test_group,
  e.event_name,
  e.event_cnt AS value
FROM events e
UNION ALL
SELECT
  s.date,
  s.country,
  s.device,
  s.continent,
  s.channel,
  s.test,
  s.test_group,
  'session' AS event_name,
  s.session_cnt AS value
FROM session s
UNION ALL
SELECT
  a.date,
  a.country,
  a.device,
  a.continent,
  a.channel,
  a.test,
  a.test_group,
  'new_account' AS event_name,
  a.new_account_cnt AS value
FROM account a
"""

df = client.query(query).to_dataframe()
df.head()

,date,country,device,continent,channel,test,test_group,event_name,value
0,2020-12-08,Palestine,desktop,Asia,Direct,4,2,new_account,1
1,2020-12-08,Palestine,desktop,Asia,Direct,3,2,new_account,1
2,2020-11-06,Puerto Rico,desktop,Americas,Social Search,2,2,new_account,1
3,2020-11-06,Puerto Rico,desktop,Americas,Social Search,1,1,new_account,1
4,2020-12-08,Croatia,desktop,Europe,Direct,4,2,new_account,1


In [ ]:
df['event_name'].unique()

array(['new_account', 'session_with_orders', 'scroll', 'user_engagement',
       'first_visit', 'page_view', 'session_start', 'view_item',
       'view_promotion', 'select_promotion', 'add_to_cart', 'select_item',
       'view_search_results', 'begin_checkout', 'add_shipping_info',
       'add_payment_info', 'click', 'session', 'view_item_list'],
      dtype=object)

#Data preparation

In [ ]:
warnings.filterwarnings('ignore')

def run_clean_analysis(df):
    results = []
    dims = ['country', 'continent', 'device', 'channel']
    target_metrics = ['add_payment_info', 'add_shipping_info', 'begin_checkout', 'new_account', 'session_with_orders']

    levels = [
        [],
        ['country'], ['continent'], ['device'], ['channel'],
        ['country', 'device'], ['continent', 'device'], ['device', 'channel'],
        ['country', 'channel'], ['continent', 'channel'],
        ['country', 'device', 'channel'], ['continent', 'device', 'channel']
    ]

    for t_id in sorted(df['test'].unique()):
        test_df = df[df['test'] == t_id]

        for subset in levels:
            group_cols = subset + ['test_group', 'event_name']
            agg = test_df.groupby(group_cols)['value'].sum().reset_index()

            table = agg.pivot_table(index=subset + ['test_group'],
                                   columns='event_name',
                                   values='value',
                                   aggfunc='sum').fillna(0).reset_index()

            if 'session' not in table.columns:
                continue

            final_groups = table.groupby(subset) if subset else [(None, table)]

            for name, group_data in final_groups:
                d1 = group_data[group_data['test_group'] == 1]
                d2 = group_data[group_data['test_group'] == 2]

                if d1.empty or d2.empty:
                    continue

                n1, n2 = d1['session'].values[0], d2['session'].values[0]

                for metric in target_metrics:
                    if metric not in table.columns:
                        continue

                    c1, c2 = d1[metric].values[0], d2[metric].values[0]

                    z_stat, p_val = 0.0, 1.0
                    if n1 > 0 and n2 > 0 and (c1 + c2) > 0:
                        try:
                            z_stat, p_val = proportions_ztest([c2, c1], [n2, n1])
                        except: pass

                    res_row = {d: 'All' for d in dims}

                    if subset:
                        if isinstance(name, tuple):
                            current_values = [str(v) for v in name]
                        else:
                            current_values = [str(name)]

                        for i, d_name in enumerate(subset):
                            res_row[d_name] = current_values[i]

                    conv1, conv2 = c1/n1 if n1 > 0 else 0, c2/n2 if n2 > 0 else 0

                    res_row.update({
                        'test': t_id,
                        'metric': metric,
                        'test_group_1_conv': conv1,
                        'test_group_2_conv': conv2,
                        'metric_change_pct': (conv2 - conv1) / conv1 if conv1 > 0 else 0,
                        'p_value': p_val,
                        'z_stat': z_stat,
                        'is_significant': p_val < 0.05
                    })
                    results.append(res_row)

    return pd.DataFrame(results)

final_table = run_clean_analysis(df)

In [ ]:
final_table.to_csv('ab_test_data.csv', index=False)

[Link to dataset](https://drive.google.com/file/d/17FSP36_cApQAB44CLPHOiiS6_IYNwWgm/view)

[Link to Tableau](https://public.tableau.com/views/AdvancedABtestingtool/ABtestingtool?:language=en-US&publish=yes&:sid=&:redirect=auth&:display_count=n&:origin=viz_share_link)